# SWASTHYAGUIDE 2.0 — 01. Heart Disease EDA

**Stage:** Dataset inspection only. No model training, no Streamlit UI.

**Scope disclaimer:** this is a research/prototype *risk-assessment* system.
It is not a medical diagnostic tool and produces no clinical claims.

**Source:** UCI Heart Disease dataset (processed Kaggle mirror),
`data/heart_disease_uci.csv` (920 rows x 16 columns).

**Read-only rule:** this notebook never writes to `data/`. The raw CSV is treated
as an immutable input. All derived tables stay in memory / notebook output.

## 1. Imports

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Reproducibility: fixed seeds + explicit library versions recorded here.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

print("pandas", pd.__version__)
print("numpy", np.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("seaborn", sns.__version__)
print("seed", RANDOM_SEED)

## 2. Load dataset

Path is resolved relative to the project root so the notebook runs from any
working directory. A SHA-256 checksum is printed to prove which file was read.

In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "heart_disease_uci.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("data/heart_disease_uci.csv not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "heart_disease_uci.csv"

print("resolved csv :", CSV_PATH)
print("size (bytes)  :", CSV_PATH.stat().st_size)
print("sha256        :", hashlib.sha256(CSV_PATH.read_bytes()).hexdigest())

# Read-only load: dtype inference left to pandas, nothing is coerced or cleaned.
df = pd.read_csv(CSV_PATH)
df.head()

## 3. Dataset overview

Shape, column names and inferred dtypes exactly as loaded from the file.
Nothing here is assumed — all of it is read off the DataFrame.

In [ ]:
print("shape (rows, cols):", df.shape)
print("columns:", list(df.columns))

schema = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().to_numpy(),
        "null": df.isna().sum().to_numpy(),
        "n_unique": df.nunique(dropna=True).to_numpy(),
    }
)
schema

In [ ]:
# Numerical columns only, for the descriptive pass in section 5.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
# Categorical = everything that is not numeric (pandas inferred them as object).
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("numeric  :", numeric_cols)
print("categorical:", categorical_cols)

### Duplicate check

`id` must be unique. Feature-level duplicates are also checked (ignoring
`id` and `dataset`) because two source cohorts can contain the same patient
profile, which would leak across a future train/test split.

In [ ]:
print("duplicate full rows          :", int(df.duplicated().sum()))
print("duplicate ids               :", int(df["id"].duplicated().sum()))

feature_cols_for_dupcheck = [c for c in df.columns if c not in ("id", "dataset")]
print("duplicate feature profiles  :", int(df.duplicated(subset=feature_cols_for_dupcheck).sum()))

## 4. Missing-value analysis

Counts and percentages per column. Note that missingness here is **not random**:
it is concentrated in specific source cohorts (shown below), which matters for
any imputation or cohort-aware split.

In [ ]:
missing = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().sum() / len(df) * 100).round(2),
    }
)
missing = missing.sort_values("missing_pct", ascending=False)
missing

print("total missing cells:", int(df.isna().sum().sum()))
print("rows fully complete :", int(df.notna().all(axis=1).sum()), "of", len(df))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax = missing[missing["missing_pct"] > 0]["missing_pct"].plot.bar(ax=ax, color="#c44e52")
ax.set_title("Missing values per column (%)")
ax.set_ylabel("% missing")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
# Missingness is strongly cohort-dependent.
miss_by_cohort = df.groupby("dataset").apply(
    lambda g: (g.isna().sum() / len(g) * 100).round(1), include_groups=False
)
miss_by_cohort.T

## 5. Numerical feature statistics

In [ ]:
df[numeric_cols].describe().T.round(3)

In [ ]:
# Physiologically implausible values observed in the raw file.
checks = {
    "trestbps == 0 (rest BP of zero)": int((df["trestbps"] == 0).sum()),
    "chol == 0 (serum chol of zero)": int((df["chol"] == 0).sum()),
    "oldpeak < 0": int((df["oldpeak"] < 0).sum()),
    "oldpeak > 5": int((df["oldpeak"] > 5).sum()),
}
pd.Series(checks, name="count")

print("chol == 0 by cohort:")
print(df.loc[df["chol"] == 0, "dataset"].value_counts().to_string())

In [ ]:
num_plot_cols = ["age", "trestbps", "chol", "thalch", "oldpeak"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, num_plot_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax, color="#4c72b0")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

## 6. Categorical feature analysis

Exact value counts as stored in the file. `fbs` and `exang` arrive as the
strings `"True"`/`"False"` inside an object column, so they need explicit
handling before encoding.

In [ ]:
for col in categorical_cols:
    counts = df[col].value_counts(dropna=False)
    print(f"\n--- {col} (dtype={df[col].dtype}, nunique={df[col].nunique()}) ---")
    print(counts.to_string())
    print(f"missing: {df[col].isna().sum()} ({df[col].isna().mean() * 100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, col in zip(axes.flat, ["sex", "cp", "restecg", "slope", "thal", "fbs"]):
    sns.countplot(data=df, x=col, ax=ax, color="#55a868", order=df[col].value_counts().index)
    ax.set_title(f"{col}")
    if df[col].nunique() <= 4:
        for container in ax.containers:
            ax.bar_label(container)
    plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

### Source cohort (`dataset`) — metadata, not a feature

`dataset` records which of the four UCI cohorts a row came from. It is not a
patient measurement and is excluded from the feature set, but it is retained
for stratified analysis and later cohort-aware splitting.

In [ ]:
cohort_sizes = df["dataset"].value_counts()
cohort_sizes

# id is a sequential row counter concatenated across cohorts, not a patient identifier.
df.groupby("dataset")["id"].agg(["min", "max", "count"]).rename(columns={"count": "n"})

## 7. Target distribution

The target column is verified from the file, not assumed. It is `num`, an
integer with 5 distinct values (0-4), i.e. **ordinal / multiclass** in the raw
form — it is the count of major vessels with narrowing in the original UCI
specification.

For the first binary experiment we evaluate the collapse `num == 0` vs `num > 0`.

In [ ]:
TARGET = "num"
print("candidate target columns present:", [c for c in ("num", "target", "label", "disease") if c in df.columns])
print("target dtype:", df[TARGET].dtype, "| n unique:", df[TARGET].nunique())
print("sorted values:", sorted(df[TARGET].dropna().unique().tolist()))

target_counts = df[TARGET].value_counts().sort_index()
target_share = (target_counts / len(df) * 100).round(2)
pd.DataFrame({"count": target_counts, "pct": target_share})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax = target_share.plot.bar(ax=ax, color="#4c72b0")
ax.set_title("Target 'num' distribution (raw, ordinal 0-4)")
ax.set_xlabel("num")
ax.set_ylabel("% of rows")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")
plt.tight_layout()
plt.show()

### Binary evaluation: `num == 0` vs `num > 0`

**Documented transformation, created in memory only:**

```python
df["target"] = df["num"].gt(0).astype(int)   # 0 = no disease, 1 = disease
```

This is a label-collapse, not a data modification — `data/heart_disease_uci.csv`
is untouched. The mapping and the resulting class balance are printed below.

In [ ]:
BINARY_TARGET = "target"
df[BINARY_TARGET] = df[TARGET].gt(0).astype(int)

binary_counts = df[BINARY_TARGET].value_counts().sort_index()
binary_share = (binary_counts / len(df) * 100).round(2)
print("label mapping: num == 0 -> 0 (no disease), num > 0 -> 1 (disease)")
pd.DataFrame({"count": binary_counts, "pct": binary_share})

print("\nbinary label dtype:", df[BINARY_TARGET].dtype, "| n unique:", df[BINARY_TARGET].nunique())
print("any NaN in target:", bool(df[BINARY_TARGET].isna().any()))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x=BINARY_TARGET, ax=ax, color="#c44e52", order=[0, 1])
ax.set_title("Binary target (num > 0) — near-balanced")
ax.set_xlabel("0 = no disease, 1 = disease")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

In [ ]:
# Target prevalence is heavily cohort-dependent -> a random split will mix
# populations of very different disease prevalence.
cohort_target = pd.crosstab(df["dataset"], df[BINARY_TARGET], margins=True)
cohort_target

print("prevalence of num>0 per cohort:")
print((df.groupby("dataset")[BINARY_TARGET].mean() * 100).round(1).to_string())

## 8. Basic visualizations

In [ ]:
# Age and max heart rate by target class.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

sns.boxplot(data=df, x=BINARY_TARGET, y="age", ax=axes[0], color="#4c72b0")
axes[0].set_title("Age by target")

sns.boxplot(data=df, x=BINARY_TARGET, y="thalch", ax=axes[1], color="#55a868")
axes[1].set_title("thalch (max heart rate) by target")

sns.boxplot(data=df, x=BINARY_TARGET, y="oldpeak", ax=axes[2], color="#c44e52")
axes[2].set_title("oldpeak by target")

plt.tight_layout()
plt.show()

In [ ]:
# Chest pain type vs target — descriptive only.
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="cp", hue=BINARY_TARGET, ax=ax, palette="deep")
ax.set_title("Chest pain type vs binary target")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation among numeric columns (pairwise, ignores NaN pairwise).
corr_cols = [c for c in numeric_cols if c != "id"]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", ax=ax, square=True)
ax.set_title("Numeric feature correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Cohort composition of the binary classes.
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="dataset", hue=BINARY_TARGET, ax=ax, palette="deep")
ax.set_title("Source cohort vs binary target")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 9. Initial observations

Facts read directly from the data above, no modelling implied:

1. **Shape** — 920 rows x 16 columns. Target column is `num`, confirmed present.
2. **Target is ordinal, not binary** — `num` takes values 0,1,2,3,4 (44.7% / 28.8% /
   11.9% / 11.6% / 3.0%). Collapsing to `num > 0` yields 411 no-disease (44.7%)
   vs 509 disease (55.3%) — a mild, workable class imbalance.
3. **Target prevalence is not comparable across cohorts** — 36.2% (Hungary) to
   93.5% (Switzerland). Pooling cohorts into one model mixes populations with very
   different base rates.
4. **Missingness is severe and cohort-structured** — `ca` 66.4% missing, `thal` 52.8%,
   `slope` 33.6%. Cleveland is nearly complete; Hungary/Switzerland/VA Long Beach
   are not. Only 299 of 920 rows are complete.
5. **Implausible physiological values exist** — `chol == 0` in 172 rows (all
   Switzerland + VA Long Beach), `trestbps == 0` in 1 row, 12 negative `oldpeak`
   values. These are almost certainly *unrecorded* measurements encoded as 0/negative,
   not real readings.
6. **Boolean columns are strings** — `fbs` and `exang` hold `"True"`/`"False"` text
   in an object column, so they need explicit mapping before encoding.
7. **`id` is a row counter, not a patient ID** — 1..920, contiguous, concatenated
   across cohorts. It carries no clinical signal and must not be a feature.
8. **`dataset` is source metadata, not a measurement** — must not be a feature.
9. **Duplicates** — no duplicate `id`, no duplicate full rows; 2 duplicate feature
   profiles when `id` and `dataset` are ignored.
10. **Class separation looks weak-to-moderate** — e.g. 63.2% of males vs 25.8% of
    females are labelled disease; `oldpeak` shows the clearest numeric shift. Small
    dataset, so expect modest signal only.

## 10. Data-cleaning decisions for the next stage

Column roles fixed now, so later code (and the FastAPI layer) can rely on them.

In [ ]:
TARGET_COL = TARGET                       # raw ordinal label, kept for reference
EXCLUDE_COLS = ["id", "dataset"]         # identifier / source metadata
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS + [TARGET_COL, BINARY_TARGET]]

print("target      :", TARGET_COL)
print("excluded    :", EXCLUDE_COLS)
print("features    :", FEATURE_COLS)
print("n_features  :", len(FEATURE_COLS))

FEATURE_ROLE = {
    "age": "numeric — patient age (years)",
    "trestbps": "numeric — resting blood pressure (mm Hg)",
    "chol": "numeric — serum cholesterol (mg/dl)",
    "thalch": "numeric — maximum heart rate achieved (bpm)",
    "oldpeak": "numeric — ST depression induced by exercise",
    "ca": "ordinal numeric — number of major vessels 0-3",
    "sex": "categorical — patient sex",
    "cp": "categorical — chest pain type (4 levels)",
    "fbs": "boolean-as-text — fasting blood sugar > 120 mg/dl",
    "restecg": "categorical — resting ECG result (3 levels)",
    "exang": "boolean-as-text — exercise induced angina",
    "slope": "categorical — peak exercise ST slope (3 levels)",
    "thal": "categorical — thalassemia test result (3 levels)",
}
assert sorted(FEATURE_ROLE) == sorted(FEATURE_COLS), "role table is out of sync with FEATURE_COLS"
pd.Series(FEATURE_ROLE).to_frame("description")

### Decisions (applied only in the modelling stage, not to the CSV)

| # | Decision | Rationale |
|---|----------|-----------|
| 1 | Never modify `data/heart_disease_uci.csv` | raw data stays reproducible; checksum recorded above |
| 2 | Drop `id` and `dataset` from features | `id` is a row counter, `dataset` is source metadata; both invite leakage |
| 3 | Build the binary label in memory as `target = (num > 0)` | documented, reversible; 44.7/55.3 split is usable |
| 4 | Keep `num` available for a later ordinal / multiclass comparison | ordinal information is not thrown away |
| 5 | Treat `chol == 0`, `trestbps == 0`, `oldpeak < 0` as missing (NaN), not valid values | physiologically impossible readings; encoding them as real data would bias the model |
| 6 | Split by cohort-aware stratification on `dataset` | prevalence ranges 36% to 94%; a plain random split would mix populations |
| 7 | Impute, do not drop, for `ca`/`thal`/`slope` | 52-66% missing; dropping would destroy most of the data. Median for numeric, explicit `"Missing"` level for categorical |
| 8 | Convert `fbs` and `exang` text to real booleans first | they are string literals in an object column |
| 9 | One-hot encode `cp`, `restecg`, `slope`, `thal`, `sex` | low cardinality, no inherent order |
| 10 | Keep `ca` as ordinal (0-3) | it is a genuine count, not an unordered category |
| 11 | Fit every transformer inside a `Pipeline` inside `ColumnTransformer` | prevents train/test leakage when FastAPI serves the model |
| 12 | Use stratified cross-validation and report ROC-AUC + PR-AUC, not accuracy alone | imbalance and threshold choice matter more than raw accuracy |

**Explicitly deferred:** no model is trained in this notebook. Nothing in `models/`
is written. Model training, evaluation and the FastAPI serving layer start only on
explicit instruction.

---

*SWASTHYAGUIDE 2.0 — research prototype. Not a medical device, not a diagnostic
system, and not approved for clinical decision-making.*